In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from xgboost import XGBRegressor

# LOAD DATA


train_data = pd.read_csv("C:/Users/ASUS/Downloads/train.csv")

print("Dataset Shape:", train_data.shape)


# TARGET VARIABLE


y = train_data["SalePrice"]

# Drop Target
X = train_data.drop(["SalePrice"], axis=1)


# SELECT FEATURES


categorical_cols = [
    col for col in X.columns
    if X[col].dtype == "object"
]

numerical_cols = [
    col for col in X.columns
    if X[col].dtype in ["int64", "float64"]
]


# PREPROCESSING


numerical_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


# MODEL


model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


# PIPELINE


pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

# TRAIN TEST SPLIT


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# TRAIN MODEL

pipeline.fit(X_train, y_train)

# PREDICTION


preds = pipeline.predict(X_valid)

# EVALUATION

mae = mean_absolute_error(y_valid, preds)
rmse = np.sqrt(mean_squared_error(y_valid, preds))
r2 = r2_score(y_valid, preds)

print("\nModel Performance")
print("------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

# FEATURE ANALYSIS

plt.figure(figsize=(10,6))
sns.histplot(train_data["SalePrice"], bins=30)
plt.title("Sale Price Distribution")
plt.show()

# TEST DATA PREDICTION

test_data = pd.read_csv("test.csv")

test_preds = pipeline.predict(test_data)

submission = pd.DataFrame({
    "Id": test_data["Id"],
    "SalePrice": test_preds
})

submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv created successfully")

ModuleNotFoundError: No module named 'xgboost'